# 15 (DE) — Table Lifecycle & Storage

**Data Engineer perspective.** Creating, replacing, appending and retiring tables: writer modes, row vs columnar storage, temp views, catalog inspection, and cleanup hygiene.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Seed data

In [ ]:
import pandas as pd

df = session.createDataFrame(pd.DataFrame({
    "sku": ["a1", "b2", "c3"],
    "estoque": [10, 20, 30],
}))
df.show()

## 2. Writer modes

`error` (default) refuses existing tables; `append` adds rows; `overwrite` replaces; `ignore` no-ops silently.

In [ ]:
df.write.saveAsTable("estoque_demo")                      # mode=error

try:
    df.write.saveAsTable("estoque_demo")                 # refuses
except Exception as e:
    print("error mode ->", str(e)[:80])

session.createDataFrame(pd.DataFrame({"sku": ["d4"], "estoque": [40]})) \
    .write.mode("append").saveAsTable("estoque_demo")
session.table("estoque_demo").show()

session.createDataFrame(pd.DataFrame({"sku": ["z9"], "estoque": [99]})) \
    .write.mode("overwrite").saveAsTable("estoque_demo")
session.table("estoque_demo").show()

session.createDataFrame(pd.DataFrame({"sku": ["never"], "estoque": [0]})) \
    .write.mode("ignore").saveAsTable("estoque_demo")
print("after ignore, skus:", [r[0] for r in session.table("estoque_demo").select("sku").collect()])

## 3. `insertInto` vs `saveAsTable`

`insertInto` appends positionally to an existing table and ignores the writer mode default.

In [ ]:
session.createDataFrame(pd.DataFrame({"sku": ["e5"], "estoque": [50]})) \
    .write.mode("append").insertInto("estoque_demo")
session.table("estoque_demo").show()

## 4. Columnar storage

`storageType('columnar')` writes a columnar table — better compression and scan performance for analytics workloads.

In [ ]:
session.createDataFrame(pd.DataFrame({
    "sensor": ["s1", "s2", "s3", "s4"],
    "temperatura": [21.5, 22.1, 20.8, 23.0],
})).write.storageType("columnar").mode("overwrite").saveAsTable("sensores_colunar_demo")

session.table("sensores_colunar_demo").show()
print("storage layout:", session.catalog.listTables())

## 5. Temp views — SQL over lazy DataFrames

In [ ]:
session.table("estoque_demo").createOrReplaceTempView("estoque_view")
session.sql("SELECT * FROM estoque_view WHERE estoque > 40").show()

## 6. Catalog inspection

In [ ]:
tables = session.catalog.listTables()
print(f"{len(tables)} tables visible:")
for t in tables[:10]:
    print(" -", t)

## 7. Cleanup hygiene

Tables from `createDataFrame` land in `SQLUSER` and are tracked on the session for cleanup; explicit demo tables are dropped here.

In [ ]:
for t in ("estoque_demo", "sensores_colunar_demo"):
    session.sql(f"DROP TABLE IF EXISTS {t}")
print("dropped demo tables")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")